# Experiment 3 - Translate-Test (Bengali -> English)

Score the **experiment 1 checkpoints unchanged** on Bengali test posts translated to English.

| Condition | Backbone | Train | Validation | Test |
|---|---|---|---|---|
| **3** translate-test | `FacebookAI/xlm-roberta-base` (**same weights as #1**) | en-train | en-val | bn-test -> en |

This experiment trains nothing: the three seeded `1_xlmr_seed{N}` checkpoints are downloaded
from experiment 1's private Hub repo and scored twice on the same held-out test items - once
in native Bengali (the reference pairing) and once in English translation. Experiment 1 vs 3
therefore differ **only** in the test representation: the cleanest pairing in the paper, and
the one that directly feeds RQ2.

The split seed is 42, matching experiments 1, 2 and 4. The three checkpoints (training seeds
42, 43, 44) measure training variability on those fixed splits.

Metrics and row-aligned predictions are pushed to a **private Hugging Face Hub repo** after
the evaluation.

In [ ]:
# Runtime check: this experiment needs a GPU.
import torch

print("torch         :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Colab select Runtime > Change runtime type > GPU.")
print("device        :", torch.cuda.get_device_name(0))

In [ ]:
# Install pinned deps. torch is deliberately NOT installed so Colab's
# CUDA build stays intact.
!pip install -q "transformers==5.18.0" accelerate openpyxl pandas scikit-learn

In [ ]:
# Get the project. Idempotent: safe to re-run after a runtime restart.
# The datasets are tracked in the repo, so the clone brings code + data.
import os, subprocess, pathlib

if not pathlib.Path("NLP").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/iammatthi/NLP.git"], check=True)
os.chdir("NLP")
print("working dir:", os.getcwd())

for rel in (
    "data/raw/depression-severity-bengali.xlsx",
    "data/translated/bn_to_en.csv",
):
    data_file = pathlib.Path(rel)
    assert data_file.exists(), f"{data_file} missing - the clone did not bring the data"
    print("dataset OK:", data_file, f"({data_file.stat().st_size:,} bytes)")

In [ ]:
# Access the private Hugging Face Hub repos: experiment 3 downloads experiment 1's
# checkpoints (read-only) and pushes its own evaluation artefacts.
# Set HF_TOKEN in Colab Secrets (key icon, left sidebar) - it is read
# from there and never printed. The token needs read access to the experiment 1
# repo. If no secret is set, you get a login prompt.
import os

HF_CHECKPOINTS = "simonfrnd/nlp_experiment1"   # experiment 3 REUSES these checkpoints
HF_REPO = "iammatthi/nlp_experiment3"          # experiment 3 writes its artefacts here
print("HF_CHECKPOINTS =", HF_CHECKPOINTS)
print("HF_REPO        =", HF_REPO)

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    print("HF_TOKEN: read from Colab Secrets")
except Exception:
    print("HF_TOKEN: no Colab Secret found, falling back to a login prompt")

from huggingface_hub import login, whoami
if token:
    login(token=token)
else:
    from huggingface_hub import notebook_login
    notebook_login()

print("authenticated as:", whoami()["name"])
print("keep the repos private: the source datasets state no licence")

In [ ]:
# CPU-only build check: verifies XLM-R loads and produces (n, 4) logits.
!python scripts/smoke_test_model.py --models FacebookAI/xlm-roberta-base

In [ ]:
# Stratified 80/10/10 splits, fixed seed. Regenerated every session; the seed
# makes them byte-reproducible so runs stay comparable.
# bn native splits carry the reference test split; the bn_to_en splits carry the
# same rows in English translation (row-aligned, verified by hash in the manifest).
!python scripts/prepare_splits.py --lang bn --seed 42
!python scripts/prepare_translated_splits.py --lang bn --translations data/translated/bn_to_en.csv --seed 42

## Get the experiment 1 checkpoints

No training happens here. The three `1_xlmr_seed{N}/final/` checkpoints (best epoch by
English validation macro-F1, ~1.1 GB each) are downloaded unchanged from experiment 1's
private repo, so experiment 1 vs 3 share **identical model weights** - only the test
representation differs.

In [ ]:
# Download the three experiment 1 checkpoints (read-only). Idempotent: safe to re-run.
from huggingface_hub import snapshot_download
import pathlib

for seed in (42, 43, 44):
    snapshot_download(
        repo_id=HF_CHECKPOINTS,
        allow_patterns=[f"1_xlmr_seed{seed}/final/*"],
        local_dir="outputs",
    )
    final = pathlib.Path(f"outputs/1_xlmr_seed{seed}/final")
    assert (final / "config.json").exists(), f"{final} missing - the download failed"
    print("checkpoint OK:", final)

## Evaluate on the held-out test split

Each checkpoint is scored twice on the same 490 test items:

- `bn-test` (native Bengali) - the experiment 1 reference, re-scored here so both
  representations come out of the same evaluation code
- `bn-test -> en` (Amazon Translate) - condition 3

Row-aligned predictions for both representations are written per seed and feed the RQ2
pairing analysis. `evaluate.py --out` keeps the scores out of the downloaded checkpoint
directories.

In [ ]:
# Native Bengali reference (identical weights, original representation)
!python -u evaluate.py --model-dir outputs/1_xlmr_seed42/final --lang bn --split test --seed 42 --out outputs/3_xlmr_seed42/eval_test_bn_native.json --predictions-out outputs/3_xlmr_seed42/predictions_bn_test_native.csv
!python -u evaluate.py --model-dir outputs/1_xlmr_seed43/final --lang bn --split test --seed 42 --out outputs/3_xlmr_seed43/eval_test_bn_native.json --predictions-out outputs/3_xlmr_seed43/predictions_bn_test_native.csv
!python -u evaluate.py --model-dir outputs/1_xlmr_seed44/final --lang bn --split test --seed 42 --out outputs/3_xlmr_seed44/eval_test_bn_native.json --predictions-out outputs/3_xlmr_seed44/predictions_bn_test_native.csv

# Condition 3 - translate-test: the same test items translated to English
!python -u evaluate.py --model-dir outputs/1_xlmr_seed42/final --lang bn --split test --splits-dir data/splits/bn_to_en --seed 42 --out outputs/3_xlmr_seed42/eval_test_bn2en.json --predictions-out outputs/3_xlmr_seed42/predictions_bn_test_en.csv
!python -u evaluate.py --model-dir outputs/1_xlmr_seed43/final --lang bn --split test --splits-dir data/splits/bn_to_en --seed 42 --out outputs/3_xlmr_seed43/eval_test_bn2en.json --predictions-out outputs/3_xlmr_seed43/predictions_bn_test_en.csv
!python -u evaluate.py --model-dir outputs/1_xlmr_seed44/final --lang bn --split test --splits-dir data/splits/bn_to_en --seed 42 --out outputs/3_xlmr_seed44/eval_test_bn2en.json --predictions-out outputs/3_xlmr_seed44/predictions_bn_test_en.csv

# metrics + row-aligned predictions are tiny - push them across too
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(HF_REPO, private=True, exist_ok=True)
api.upload_folder(
    repo_id=HF_REPO,
    folder_path="outputs",
    allow_patterns=["3_xlmr_seed*/*"],
    commit_message="Add experiment 3 evaluation artefacts",
)
print("pushed experiment 3 artefacts to", HF_REPO)

## Summarise across seeds

In [ ]:
import glob, json, statistics as st
import pandas as pd

def mean_std(vals):
    m = st.mean(vals)
    s = st.stdev(vals) if len(vals) > 1 else 0.0
    return f"{m:.4f} +/- {s:.4f}"

native = [json.load(open(p)) for p in sorted(glob.glob("outputs/3_xlmr_seed*/eval_test_bn_native.json"))]
bn2en = [json.load(open(p)) for p in sorted(glob.glob("outputs/3_xlmr_seed*/eval_test_bn2en.json"))]
assert native and bn2en, "no results - run the evaluation cells first"
print(f"3_xlmr  n={len(bn2en)} (translate-test, same weights as experiment 1)")
print(f"{'metric':20s} {'bn-test (native)':22s} {'bn-test -> en':22s}")
for k in ("macro_f1", "qwk", "mae", "accuracy"):
    print(f"  {k:18s} {mean_std([r[k] for r in native]):22s} {mean_std([r[k] for r in bn2en]):22s}")
for k in ("upgrade_rate", "downgrade_rate", "error_rate", "mean_signed_error"):
    print(f"  {k:18s} {mean_std([r['direction'][k] for r in native]):22s} {mean_std([r['direction'][k] for r in bn2en]):22s}")

print("\nper-class F1 (mean across seeds):")
for cls in ("minimal", "mild", "moderate", "severe"):
    print(f"  {cls:9s} {mean_std([r[cls] for r in native]):22s} {mean_std([r[cls] for r in bn2en]):22s}")

# paired per-item comparison: same rows, two representations (feeds RQ2)
print("\npaired predictions (same 490 items):")
for seed in (42, 43, 44):
    a = pd.read_csv(f"outputs/3_xlmr_seed{seed}/predictions_bn_test_native.csv")
    b = pd.read_csv(f"outputs/3_xlmr_seed{seed}/predictions_bn_test_en.csv")
    assert a["label"].tolist() == b["label"].tolist(), f"seed {seed}: rows misaligned"
    same = int((a["prediction"] == b["prediction"]).sum())
    print(f"  seed {seed}: {same}/{len(a)} unchanged, {len(a) - same} changed under translation")

In [ ]:
# Export metrics to the local machine. Checkpoints live on the Hub, this zip is just
# the small numbers: both evaluations per seed plus the row-aligned predictions.
import glob, zipfile

paths = sorted(
    glob.glob("outputs/3_xlmr_seed*/eval_test_*.json")
    + glob.glob("outputs/3_xlmr_seed*/predictions_*.csv")
)
assert paths, "no results found - run the evaluation cells first"

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p)

print(f"zipped {len(paths)} files:")
for p in paths:
    print(" ", p)

from google.colab import files
files.download("results.zip")
print("downloaded results.zip")

In [ ]:
# What ended up on the HF. Expect 12 small files: two evaluations + row-aligned
# predictions per seed. No model weights - the checkpoints stay in experiment 1's repo.
from huggingface_hub import HfApi
api = HfApi()
info = api.repo_info(HF_REPO, files_metadata=True)
print(f"repo: {info.id}  (private={info.private})")
total = 0
for s in sorted(info.siblings, key=lambda s: s.rfilename):
    size = getattr(s, "size", 0) or 0
    total += size
    print(f"  {size/1e6:8.1f} MB  {s.rfilename}")
print(f"  total {total/1e6:.1f} MB")